# Lab 2: Building an Analytical Base Table for Machine Learning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
FIGURES_DIR = Path("../figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
for file in DATA_DIR.glob("*.csv"):
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [4]:
orders = pd.read_csv(DATA_DIR/"olist_orders_dataset.csv")

customers = pd.read_csv(DATA_DIR/"olist_customers_dataset.csv")

reviews = pd.read_csv(DATA_DIR/"olist_order_reviews_dataset.csv")

payments = pd.read_csv(DATA_DIR/"olist_order_payments_dataset.csv")

order_items = pd.read_csv(DATA_DIR/"olist_order_items_dataset.csv")

products = pd.read_csv(DATA_DIR/"olist_products_dataset.csv")

category_translation = pd.read_csv(
    DATA_DIR/"product_category_name_translation.csv"
)

In [5]:
tables = {
    "orders": orders,
    "customers": customers,
    "reviews": reviews,
    "payments": payments,
    "order_items": order_items,
    "products": products,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

orders: 99441 rows, 8 columns
customers: 99441 rows, 5 columns
reviews: 99224 rows, 7 columns
payments: 103886 rows, 5 columns
order_items: 112650 rows, 7 columns
products: 32951 rows, 9 columns
category_translation: 71 rows, 2 columns


In [6]:
orders.head()

customers.head()

payments.head()

order_items.head()

reviews.head()

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [7]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [8]:
orders["order_purchase_timestamp"].dt.year

0        2017
1        2018
2        2018
3        2017
4        2018
         ... 
99436    2017
99437    2018
99438    2017
99439    2018
99440    2018
Name: order_purchase_timestamp, Length: 99441, dtype: int32

In [ ]:
orders["order_year"] = orders["order_purchase_timestamp"].dt.year

orders["order_month"] = orders["order_purchase_timestamp"].dt.month

orders["order_day"] = orders["order_purchase_timestamp"].dt.day

orders["order_day_of_week"] = orders["order_purchase_timestamp"].dt.dayofweek

orders["order_hour"] = orders["order_purchase_timestamp"].dt.hour

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    -
    orders["order_purchase_timestamp"]
).dt.days

In [ ]:
orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"]
    -
    orders["order_purchase_timestamp"]
).dt.days

In [ ]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [ ]:
orders["is_late_delivery"] = np.where(
    orders["delivery_delay_days"] > 0,
    1,
    0
)

In [ ]:
orders["is_late_delivery"].value_counts().plot(kind="bar")

In [ ]:
payment_features = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    max_payment_installments=("payment_installments", "max"),
    payment_types_count=("payment_type", "nunique")
).reset_index()

In [ ]:
total_payment_value=("payment_value","sum")

In [ ]:
dominant_payment_type = payments.groupby("order_id")["payment_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).reset_index()

In [ ]:
payment_types_count=("payment_type","nunique")

In [ ]:
dominant_payment_type = dominant_payment_type.rename(
    columns={"payment_type":"dominant_payment_type"}
)

In [13]:
print(order_items.columns)

Index(['order_id', '                   "order_item_id"', ' "product_id"',
       '             "seller_id"', '               "shipping_limit_date"',
       '   "price"', ' "freight_value"'],
      dtype='str')


In [14]:
order_items.columns = (
    order_items.columns
    .str.strip()        # extra spaces hatao
    .str.replace('"', '', regex=False)   # quotes hatao
)

print(order_items.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')


In [15]:
item_features = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique")
).reset_index()

In [16]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [18]:
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

In [20]:
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [21]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [22]:
print(order_items.columns.tolist())

['order_id', '                   "order_item_id"', ' "product_id"', '             "seller_id"', '               "shipping_limit_date"', '   "price"', ' "freight_value"']


In [23]:
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [24]:
with open(DATA_DIR/"olist_order_items_dataset.csv", "r", encoding="utf-8") as f:
    print(f.readline())

"order_id",                   "order_item_id", "product_id",             "seller_id",               "shipping_limit_date",   "price", "freight_value"



In [25]:
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

order_items.columns = (
    order_items.columns
    .str.strip()                 # Leading/trailing spaces hatao
    .str.replace('"', '', regex=False)  # Double quotes hatao
)

print(order_items.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [26]:
item_features = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique")
).reset_index()

In [27]:
items_products = order_items.merge(
    products,
    on="product_id",
    how="left"
)

In [28]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.replace('"', '', regex=False)
    )
    return df

orders = clean_columns(orders)
customers = clean_columns(customers)
reviews = clean_columns(reviews)
payments = clean_columns(payments)
order_items = clean_columns(order_items)
products = clean_columns(products)
category_translation = clean_columns(category_translation)

In [31]:
payment_features = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    max_payment_installments=("payment_installments", "max"),
    payment_types_count=("payment_type", "nunique")
).reset_index()

In [32]:
payment_features

,order_id,total_payment_value,max_payment_installments,payment_types_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1
99437,fffce4705a9662cd70adb13d4a31832d,116.85,3,1
99438,fffe18544ffabc95dfada21779c9644f,64.71,3,1


In [33]:
abt = orders.merge(
    payment_features,
    on="order_id",
    how="left"
)

In [34]:
dominant_payment_type = payments.groupby("order_id")["payment_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
).reset_index()

dominant_payment_type = dominant_payment_type.rename(
    columns={"payment_type": "dominant_payment_type"}
)

payment_features = payment_features.merge(
    dominant_payment_type,
    on="order_id",
    how="left"
)

In [35]:
payment_features.head()

,order_id,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,credit_card


In [36]:
abt = orders.merge(
    payment_features,
    on="order_id",
    how="left"
)

In [37]:
print(abt.shape)

(99441, 12)


In [38]:
abt.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,1.0,2.0,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,3.0,1.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,1.0,1.0,credit_card


In [39]:
abt.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
total_payment_value                 1
max_payment_installments            1
payment_types_count                 1
dominant_payment_type               1
dtype: int64

In [40]:
abt.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   total_payment_value            99440 non-null  float64       
 9   max_payment_installments       99440 non-null  float64       
 10  payment_types_count            99440 non-null  float64       
 11  dominant_payment_type     

In [42]:
print(abt.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type']


In [43]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [44]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [45]:
orders[["order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


In [46]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [47]:
abt = orders.merge(
    payment_features,
    on="order_id",
    how="left"
)

In [50]:
items_products = order_items.merge(
    products,
    on="product_id",
    how="left"
)


In [51]:
items_products = items_products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [52]:
order_category = items_products.groupby("order_id").agg(
    main_product_category=(
        "product_category_name_english",
        lambda x: x.mode().iloc[0] if not x.mode().empty else None
    )
).reset_index()

In [54]:
order_category.head()

,order_id,main_product_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools


In [55]:
abt = abt.merge(
    order_category,
    on="order_id",
    how="left"
)

In [56]:
abt.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_days                    2965
delivery_delay_days              2965
total_payment_value                 1
max_payment_installments            1
payment_types_count                 1
dominant_payment_type               1
total_items                       775
total_price                       775
total_freight                     775
unique_products                   775
unique_sellers                    775
main_product_category            2185
dtype: int64

In [57]:
abt["delivery_days"] = abt["delivery_days"].fillna(
    abt["delivery_days"].median()
)

In [58]:
abt["main_product_category"] = abt[
    "main_product_category"
].fillna("Unknown")

In [59]:
print(abt.duplicated().sum())

0


In [60]:
abt = abt.drop_duplicates()

In [61]:
abt.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   delivery_days                  99441 non-null  float64       
 9   delivery_delay_days            96476 non-null  float64       
 10  total_payment_value            99440 non-null  float64       
 11  max_payment_installments  

In [62]:
abt.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,total_payment_value,max_payment_installments,payment_types_count,total_items,total_price,total_freight,unique_products,unique_sellers
count,99441,99281,97658,96476,99441,99441.000000,96476.000000,99440.000000,99440.000000,99440.000000,98666.000000,98666.000000,98666.000000,98666.000000,98666.000000
mean,2017-12-31 08:43:12.776581,2017-12-31 18:35:24.098800,2018-01-04 21:49:48.138278,2018-01-14 12:09:19.035542,2018-01-24 03:08:37.730111,12.031647,-11.876881,160.990267,2.930521,1.022586,1.141731,137.754076,22.823562,1.038098,1.013622
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,0.000000,-147.000000,0.000000,0.000000,1.000000,1.000000,0.850000,0.000000,1.000000,1.000000
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.250000,2017-09-25 22:07:22.250000,2017-10-03 00:00:00,6.000000,-17.000000,62.010000,1.000000,1.000000,1.000000,45.900000,13.850000,1.000000,1.000000
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,10.000000,-12.000000,105.290000,2.000000,1.000000,1.000000,86.900000,17.170000,1.000000,1.000000
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.250000,2018-05-25 00:00:00,15.000000,-7.000000,176.970000,4.000000,1.000000,1.000000,149.900000,24.040000,1.000000,1.000000
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,209.000000,188.000000,13664.080000,24.000000,2.000000,21.000000,13440.000000,1794.960000,8.000000,5.000000
std,NaN,NaN,NaN,NaN,NaN,9.415006,10.183854,221.951257,2.715685,0.148582,0.538452,210.645145,21.650909,0.226456,0.122297


In [63]:
abt.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery


In [66]:
abt.to_csv(
    PROCESSED_DIR / "analytical_base_table.csv",
    index=False
)

In [69]:
abt.to_csv(
    PROCESSED_DIR / "analytical_base_table.csv",
    index=False
)

In [70]:
abt_check = pd.read_csv(
    PROCESSED_DIR / "analytical_base_table.csv"
)

In [71]:
print(abt.shape)
print(abt_check.shape)

(99441, 20)
(99441, 20)


In [72]:
print(abt.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category']
